# Modèle Centralisé — Baseline

Entraîne BiLSTM, ResNet1D et InceptionTime sur **toutes les données réunies**.
Ces résultats sont la **référence supérieure** pour évaluer le FL.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

DATA_SRC = os.path.join(os.path.expanduser('~'), 'Downloads',
                        'federated_projet2A_data', 'federated_projet2A')
sys.path.insert(0, DATA_SRC)

from utils.utils import load_dataset_v1, generate_data_for_train
from modeles.lstm import BiLSTMNet, GaitDataset
from modeles.resnet import ResNet1D
from modeles.inceptiontime import InceptionTime

DEVICE = torch.device('cpu')
print(f'Device : {DEVICE}')

## 1. Chargement des données

In [ ]:
dset_dir = os.path.join(DATA_SRC, 'archives', 'AQM', 'TDvsPC_new_no_shuffle/')
x_list, _ = load_dataset_v1(dset_dir, ['CP.npz', 'TD.npz'], channel_first=True, flatten=False)
idx_files = [os.path.join(DATA_SRC, 'DataCP - all.csv'),
             os.path.join(DATA_SRC, 'DataTD - By Subject.csv')]
x_tr, x_te, y_tr, y_te, x_val, y_val, ends, X_d, y_d = \
    generate_data_for_train(x_list, idx_files, order=[1, 0])

def cycles(pids, ends, X_d, y_d):
    Xl, yl = [], []
    for p in pids:
        s = int(ends[p-1]) if p > 0 else 0
        Xl.append(X_d[s:int(ends[p])])
        yl.append(y_d[s:int(ends[p])])
    return np.concatenate(Xl), np.concatenate(yl)

Xtr, Ytr = cycles(x_tr,  ends, X_d, y_d)
Xva, Yva = cycles(x_val, ends, X_d, y_d)
Xte, Yte = cycles(x_te,  ends, X_d, y_d)
print(f'Train {Xtr.shape} | Val {Xva.shape} | Test {Xte.shape}')

In [ ]:
B = 32
tr_ld = DataLoader(GaitDataset(Xtr, Ytr), batch_size=B, shuffle=True)
va_ld = DataLoader(GaitDataset(Xva, Yva), batch_size=B)
te_ld = DataLoader(GaitDataset(Xte, Yte), batch_size=B)

## 2. Boucle d'entraînement

In [ ]:
def train_model(model, tr, va, epochs=50, lr=1e-3, tag=''):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
    hist = {'tl': [], 'vl': [], 'va': []}
    for ep in range(1, epochs+1):
        model.train()
        tl = 0
        for X, y in tr:
            opt.zero_grad()
            loss = F.cross_entropy(model(X.to(DEVICE)), y.to(DEVICE))
            loss.backward(); opt.step()
            tl += loss.item() * len(X)
        sch.step()
        model.eval(); vl, ok, tot = 0, 0, 0
        with torch.no_grad():
            for X, y in va:
                out = model(X.to(DEVICE))
                vl += F.cross_entropy(out, y.to(DEVICE)).item() * len(X)
                ok += (out.argmax(1) == y.to(DEVICE)).sum().item(); tot += len(X)
        hist['tl'].append(tl/len(tr.dataset)); hist['vl'].append(vl/tot); hist['va'].append(ok/tot)
        if ep % 10 == 0:
            print(f'[{tag}] ep{ep:3d} | train={tl/len(tr.dataset):.4f} | val_acc={ok/tot:.4f}')
    return model, hist

def evaluate(model, loader, tag=''):
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds += model(X.to(DEVICE)).argmax(1).cpu().tolist(); trues += y.tolist()
    print(f'\n=== {tag} ==='); print(classification_report(trues, preds, target_names=['TD','CP']))
    return accuracy_score(trues, preds)

## 3. BiLSTM (priorité)

In [ ]:
bilstm, h1 = train_model(BiLSTMNet(), tr_ld, va_ld, epochs=50, tag='BiLSTM')
acc1 = evaluate(bilstm, te_ld, 'BiLSTM Centralisé')

## 4. ResNet1D

In [ ]:
resnet, h2 = train_model(ResNet1D(), tr_ld, va_ld, epochs=50, tag='ResNet1D')
acc2 = evaluate(resnet, te_ld, 'ResNet1D Centralisé')

## 5. InceptionTime

In [ ]:
inception, h3 = train_model(InceptionTime(), tr_ld, va_ld, epochs=50, tag='InceptionTime')
acc3 = evaluate(inception, te_ld, 'InceptionTime Centralisé')

## 6. Résumé et sauvegarde

In [ ]:
print('\n=== RÉSUMÉ CENTRALISÉ ===')
results = {'BiLSTM': acc1, 'ResNet1D': acc2, 'InceptionTime': acc3}
for k, v in results.items(): print(f'  {k:<14}: {v:.4f}')

os.makedirs(os.path.join(PROJECT_ROOT, 'resultats'), exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, h) in zip(axes, [('BiLSTM', h1), ('ResNet1D', h2), ('InceptionTime', h3)]):
    ax.plot(h['tl'], label='Train loss'); ax.plot(h['vl'], label='Val loss')
    ax2 = ax.twinx(); ax2.plot(h['va'], 'g--', label='Val acc')
    ax.set_title(name); ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'resultats', 'centralise_courbes.png'), dpi=150)
plt.show()

torch.save(bilstm.state_dict(),    os.path.join(PROJECT_ROOT, 'resultats', 'bilstm_centralise.pt'))
torch.save(resnet.state_dict(),    os.path.join(PROJECT_ROOT, 'resultats', 'resnet_centralise.pt'))
torch.save(inception.state_dict(), os.path.join(PROJECT_ROOT, 'resultats', 'inception_centralise.pt'))
print('Modèles sauvegardés dans resultats/')